<a href="https://colab.research.google.com/github/GreatLakesCommission/IEDRR_inland_lakes/blob/main/notebooks/1_get_spp_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Script to request, QA/QC and combine invasive species observations from several sources.
Run the setup blocks, the blocks for the sources you want to pull from, then the combine block.

Requesting data from GBIF and EDDMapS requires that you have an account with them.


*   To use the GBIF code block, add your GBIF username and password and an email address for notifications to Secrets in your copy of this notebook as GBIF_USER, GBIF_PWD and GBIF_EMAIL.
*   To use the EDDMaps block, add your account credentials as EDDMAPS_USER and EDDMAPS_PWD.




# Setup Blocks

In [ ]:
# general setup
%%capture
from google.colab import drive
drive.mount('/content/drive')
import datetime
import os
os.chdir("drive/My Drive/iedrr")
today = datetime.date.today().strftime('%Y%m%d')
outfolder = "speciesobs_"+today
if not os.path.exists(outfolder):
  os.mkdir(outfolder)
os.chdir(outfolder)
import pandas as pd
!pip install pyreadr
import pyreadr
from scipy import spatial
from google.colab import userdata



In [ ]:
import glob
from time import sleep
import zipfile
import math

In [ ]:
from datetime import date

In [ ]:
# only grab obs collected during this year or later
startyear = 2024

In [ ]:
os.chdir("/content/drive/My Drive/iedrr")

In [ ]:
lakes_env = pd.read_csv('final_inputs_Jan2025/lakes_env_imputed.csv')

In [ ]:
#today = "20260208"
today = date.today().strftime("%Y%m%d")


In [ ]:

outfolder = "/content/drive/My Drive/iedrr/speciesobs_"+today
os.chdir(outfolder)

In [ ]:
# run for bounding box containing the Great Lakes states or for the states themselves?
extent = "states" # options: states, box

# invasive species of interest, scientific names are formatted as lists to make it
# possible to include alternate names where more than one is in use across the
# regional databases

fishlist = [
    [['Channa'], 'Snakeheads'],
    [['Clarias batrachus'], 'Walking catfish'],
    [['Gymnocephalus cernua'], 'Ruffe'],
    [['Misgurnus anguillicaudatus'], 'Pond loach'],
    [['Neogobius melanostomus'], 'Round goby'],
    [['Osmerus mordax'], 'Rainbow smelt'],
    [['Osteoglossum bicirrhosum'], 'Silver arowana'],
    [['Proterorhinus semilunaris'], 'Tubenose goby'],
    [['Tinca tinca'], 'Tench'],
]

plantlist = [
    [['Alternanthera philoxeroides'], 'Alligator weed'],
    [['Cabomba caroliniana'], 'Carolina fanwort'],
    [['Callitriche stagnalis'], 'Pond water-starwort'],
    [['Elodea densa', 'Egeria densa'], 'Brazilian waterweed'],
    [['Hottonia palustris'], 'Water violet'],
    [['Hydrilla verticillata'], 'Hydrilla'],
    [['Hydrocharis morsus-ranae'], 'European frog-bit'],
    [['Hygrophila polysperma'], 'Indian swampweed'],
    [['Limnophila sessiliflora'], 'Dwarf ambulia'],
    [['Ludwigia grandiflora'], 'Large-flower primrose-willow'],
    [['Ludwigia hexapetala'], 'Six petal water primrose'],
    [['Ludwigia peploides'], 'Creeping water primrose'],
    [['Marsilea mutica'], 'Australian water-clover'],
    [['Marsilea quadrifolia'], 'European water-clover'],
    [['Myriophyllum aquaticum'], 'Parrot feather'],
    [['Najas minor'], 'Brittle naiad'],
    [['Nasturtium officinale'], 'Water-cress'],
    [['Nelumbo nucifera'], 'Sacred lotus'],
    [['Nitellopsis obtusa'], 'Starry stonewort'],
    [['Ottelia alismoides'], 'Duck-lettuce'],
    [['Pistia stratiotes'], 'Water lettuce'],
    [['Pontederia azurea', 'Eichhornia azurea'], 'Anchored water-hyacinth'],
    [['Pontederia crassipes', 'Eichhornia crassipes'],'Common water-hyacinth'],
    [['Sagittaria sagittifolia'], 'Hawaii arrowhead'],
    [['Salvinia auriculata'], 'Eared salvinia'],
    [['Salvinia molesta'], 'Giant salvinia'],
    [['Spirodela punctata'], 'Dotted duckweed'],
    [['Stratiotes aloides'], 'Water soldier'],
    [['Trapa natans'], 'European water chestnut']
]


# plantlist_slim doesn't include emergent species
plantlist_slim = [
    [['Cabomba caroliniana'], 'Carolina fanwort'],
    [['Callitriche stagnalis'], 'Pond water-starwort'],
    [['Elodea densa', 'Egeria densa'], 'Brazilian waterweed'],
    [['Hydrilla verticillata'], 'Hydrilla'],
    [['Hydrocharis morsus-ranae'], 'European frog-bit'],
    [['Hygrophila polysperma'], 'Indian swampweed'],
    [['Limnophila sessiliflora'], 'Dwarf ambulia'],
    [['Marsilea mutica'], 'Australian water-clover'],
    [['Marsilea quadrifolia'], 'European water-clover'],
    [['Myriophyllum aquaticum'], 'Parrot feather'],
    [['Najas minor'], 'Brittle naiad'],
    [['Nelumbo nucifera'], 'Sacred lotus'],
    [['Nitellopsis obtusa'], 'Starry stonewort'],
    [['Ottelia alismoides'], 'Duck-lettuce'],
    [['Pistia stratiotes'], 'Water lettuce'],
    [['Pontederia azurea', 'Eichhornia azurea'], 'Anchored water-hyacinth'],
    [['Pontederia crassipes', 'Eichhornia crassipes'],'Common water-hyacinth'],
    [['Sagittaria sagittifolia'], 'Hawaii arrowhead'],
    [['Salvinia auriculata'], 'Eared salvinia'],
    [['Salvinia molesta'], 'Giant salvinia'],
    [['Spirodela punctata'], 'Dotted duckweed'],
    [['Stratiotes aloides'], 'Water soldier'],
    [['Trapa natans'], 'European water chestnut']
]

invertlist = [
    [['Bithynia tentaculata'], 'Faucet snail'],
    [['Bythotrephes longimanus'], 'Spiny water flea'],
    [['Cercopagis pengoi'], 'Fishhook waterflea'],
    [['Corbicula fluminea'], 'Basket clam'],
    [['Dreissena bugensis'], 'Quagga mussel'],
    [['Dreissena polymorpha'], 'Zebra mussel'],
    [['Eriocheir sinensis'], 'Mitten crab'],
    [['Hemimysis anomala'], 'Bloody red shrimp'],
    [['Melanoides tuberculata'], 'Red-rimmed melania'],
    [['Potamopyrgus antipodarum'], 'New Zealand mud snail'],
    [['Procambarus virginalis'],'Marbled crayfish (Marmorkrebs)']
]



fishlist = pd.DataFrame(fishlist, columns=['sci_name', 'common_name'])
plantlist = pd.DataFrame(plantlist, columns=['sci_name', 'common_name'])
plantlist_slim = pd.DataFrame(plantlist_slim, columns=['sci_name', 'common_name'])
invertlist = pd.DataFrame(invertlist, columns=['sci_name', 'common_name'])

fishlist['taxon'] = 'fish'
plantlist['taxon'] = 'plant'
plantlist_slim['taxon'] = 'plant'
invertlist['taxon'] = 'invertebrate'



my_vars = {}

my_vars["fish"] = fishlist
my_vars["plant"] = plantlist
my_vars["invert"] = invertlist


# get institution lat/longs for QAQC
url = "https://github.com/ropensci/CoordinateCleaner/raw/refs/heads/master/data/institutions.rda"
dst_path = os.path.join(os.getcwd(), "institutions.rda") # download to CoLab
res = pyreadr.read_r(pyreadr.download_file(url, dst_path)) # convert rda to dictionary of dataframes

institutions = res["institutions"]
# filter to institutions within GL bounding box
institutions = institutions[(institutions['decimalLatitude'].between(36.9171,49.6117)) & (institutions['decimalLongitude'].between(-100.5513,-71.79))]
# set up a k-dimensional tree
inst_coords = list(zip(institutions["decimalLatitude"], institutions["decimalLongitude"]))
tree = spatial.KDTree(inst_coords)

def calculate_min(row):
    return tree.query([(row["lat_dec"],row["lon_dec"])])[0][0]

!rm institutions.rda

# GBIF

In [ ]:
#GBIF setup
#!pip list
!pip install pygbif
# capture suppresses output


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.6 MB/s eta 0:00:00


In [ ]:

from pygbif import species as species
from pygbif import occurrences as occ
from pygbif.occurrences.download import GbifDownload


# add your gbif.org username, password and contact email for download notices to Colab's 'Secrets'
# toggle notebook access on for all three
%env GBIF_USER=userdata.get('GBIF_USER')
%env GBIF_PWD = userdata.get('GBIF_PWD')
%env GBIF_EMAIL = userdata.get('GBIF_EMAIL')

SLEEP_DURATION = 20


# download GBIF species obs since 1970 within GL bounding box as zip files via API


#skip = ["fish", "plant", "invert"] #skip taxa you previously exported a csv for
skip = []

# get GBIF taxon IDs based on scientific names
def getskey(z):
  #return species.name_backbone(z)['usageKey']
  return species.name_backbone(z)['usage']['key']

for i, (k, v) in enumerate(my_vars.items()):
  if not k in skip:
    records = []
    print("downloading ", k)
    splist = v['sci_name'].sum() # make list of all species inc. alternate scientific names
    #splist = list(set(splist))
    spkeys = [ getskey(x) for x in splist ]
    spkeys = list(map(str, spkeys))

    # construct query
    gbif_query = GbifDownload(userdata.get('GBIF_USER'), userdata.get('GBIF_EMAIL'))
    gbif_query.add_predicate_dict({"type": "in", "key": "BASIS_OF_RECORD", "values": ['HUMAN_OBSERVATION', 'OBSERVATION', 'MACHINE_OBSERVATION', 'LIVING_SPECIMEN', 'MATERIAL_SAMPLE'], "matchCase": "false"})
    gbif_query.add_predicate_dict({"type": "equals", "key": 'HAS_COORDINATE', 'value': 'TRUE', "matchCase": "false"})
    gbif_query.add_predicate_dict({"type": "equals", "key": 'HAS_GEOSPATIAL_ISSUE', 'value': 'FALSE', "matchCase": "false"})
    gbif_query.add_predicate_dict({"type": "within", "geometry": "POLYGON((-100.551 36.917,-71.79 36.917,-71.79 49.612,-100.551 49.612,-100.551 36.917))"})
    gbif_query.add_predicate_dict({"type": "greaterThanOrEquals", "key": 'YEAR', 'value': startyear, "matchCase": "false"})
    gbif_query.add_predicate_dict({"type": "in", "key": 'TAXON_KEY', 'values': spkeys, "matchCase": "false"})
    # submit download query
    xx = gbif_query.post_download(userdata.get('GBIF_USER'), userdata.get('GBIF_PWD'))
    # wait for download to be ready
    while True:
      print(f"waiting to get download {xx}...")
      status = occ.download_meta(key = xx)['status']

      if status not in ['PREPARING', 'RUNNING']:  # = not ready yet
          if status == 'SUCCEEDED':
              print(f"Download is ready, getting it")
              output_path = k+"_gbif_obs"
              if os.path.exists(output_path): # get rid of any previous downloads for this run
                files = glob.glob(output_path+'/*.zip')
                for f in files:
                  os.remove(f)
              else:
                os.mkdir(output_path)

              occ.download_get(xx, output_path)
          else:
              print(f"Status is {status}, why?")
              print(occ.download_meta(key = xx))
          break

      sleep(SLEEP_DURATION)

    print("finished with ", k)


env: GBIF_USER=userdata.get('GBIF_USER')
env: GBIF_PWD=userdata.get('GBIF_PWD')
env: GBIF_EMAIL=userdata.get('GBIF_EMAIL')
downloading  fish
waiting to get download 0007928-260221153910048...
waiting to get download 0007928-260221153910048...
waiting to get download 0007928-260221153910048...
waiting to get download 0007928-260221153910048...
waiting to get download 0007928-260221153910048...
waiting to get download 0007928-260221153910048...
waiting to get download 0007928-260221153910048...
waiting to get download 0007928-260221153910048...
waiting to get download 0007928-260221153910048...
waiting to get download 0007928-260221153910048...
waiting to get download 0007928-260221153910048...
waiting to get download 0007928-260221153910048...
waiting to get download 0007928-260221153910048...
waiting to get download 0007928-260221153910048...
waiting to get download 0007928-260221153910048...
waiting to get download 0007928-260221153910048...
waiting to get download 0007928-26022115391

TimeoutException: Requesting secret GBIF_USER timed out. Secrets can only be fetched when running from the Colab UI.

In [ ]:

gbif_obs = {}
gbif_observers = {}
# read the GBIF data back in and QA/QC it
for k in my_vars:
  # read zip file into dataframe
  output_path = k+"_gbif_obs"
  files = os.listdir(output_path)
  file_path = os.path.join(output_path, files[0])
  print(file_path)
  base_name, extension = os.path.splitext(files[0])
  zf = zipfile.ZipFile(file_path)
  df = pd.read_csv(zf.open(base_name+'.csv'), sep='\t')
  # drop absences
  df = df.loc[df['occurrenceStatus'] == "PRESENT"]


  print(len(df.index), k, " records")
  gbif_observers[k] = df['recordedBy'].unique()

  df = df[['gbifID', 'species', 'eventDate', 'decimalLatitude', 'decimalLongitude']]
  df.rename(columns={'gbifID':'uid', "species":"sci_name", "eventDate":"obs_date", "decimalLatitude":"lat_dec", "decimalLongitude":"lon_dec"}, inplace=True)
  df["source"] = "GBIF"
  df['obs_date'] = df['obs_date'].str.slice(0, 10)
  df['obs_date'] = pd.to_datetime(df['obs_date'], format='mixed')
  # Drop rows with invalid dates
  df = df.dropna(subset='obs_date')
  df.reset_index(drop=True,inplace=True)
  # export to csv before continuing because this cell will take forever
  df.to_csv(output_path + '/'+ k +'_obs_gbifraw_' + today + '.csv', index=False)
  df_obs = pd.DataFrame(gbif_observers[k])
  df_obs.to_csv(output_path + '/'+ k +'_observers_gbif_' + today + '.csv', index=False)
  # add to dict
  gbif_obs[k] = df



fish_gbif_obs/0001102-260208012135463.zip


/tmp/ipython-input-3965930109.py:12: DtypeWarning: Columns (17,29,36,37,38,39,40,41,43,44,46,48) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(zf.open(base_name+'.csv'), sep='\t')


50844 fish  records
plant_gbif_obs/0001137-260208012135463.zip


/tmp/ipython-input-3965930109.py:12: DtypeWarning: Columns (39) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(zf.open(base_name+'.csv'), sep='\t')


19456 plant  records
invert_gbif_obs/0001145-260208012135463.zip
11281 invert  records


In [ ]:
# if starting here: resurrect from csv
gbif_obs = {}

for k in my_vars:
  output_path = k+"_gbif_obs"
  in_csv = output_path + '/'+ k +'_obs_gbifraw_' + today + '.csv'
  # add to dict
  gbif_obs[k] = pd.read_csv(in_csv)

In [ ]:
# download potential pseudoabsences
skip = ["fish"]

for i, (k, v) in enumerate(my_vars.items()):
  if not k in skip:

    # get progress from previous runs
    if 'done' in locals():
      del done
    try:
      done = pabs[k]['identifiedBy'].iloc[-1]
    except:
      print('starting')

    records = []
    print("downloading ", k)
    # clean nans
    observers = gbif_observers[k]
    observers = [x for x in observers if x == x]
    observers.sort(key=str.lower)


    # too many observers, break into chunks
    start = 0
    end = len(observers)
    step = 50
    steps = math.ceil(end/step)
    for x in range(start, end, step):
      observers_chunk = observers[x:x+step]
      # skip already processed
      if 'done' in locals():
        if done > observers_chunk[0]:
          continue
      print(f'chunk {x}/{steps}: ', observers_chunk[0])
      # construct query
      gbif_query = GbifDownload(userdata.get('GBIF_USER'), userdata.get('GBIF_EMAIL'))
      gbif_query.add_predicate_dict({"type": "equals", "key": 'HAS_COORDINATE', 'value': 'TRUE', "matchCase": "false"})
      gbif_query.add_predicate_dict({"type": "equals", "key": 'HAS_GEOSPATIAL_ISSUE', 'value': 'FALSE', "matchCase": "false"})
      gbif_query.add_predicate_dict({"type": "within", "geometry": "POLYGON((-100.551 36.917,-71.79 36.917,-71.79 49.612,-100.551 49.612,-100.551 36.917))"})
      gbif_query.add_predicate_dict({"type": "greaterThanOrEquals", "key": 'YEAR', 'value': startyear, "matchCase": "false"})
      gbif_query.add_predicate_dict({"type": "in", "key": 'RECORDED_BY', 'values': observers_chunk, "matchCase": "false"})
      # limit results to more or less this type of organism
      if k == 'fish':
        gbif_query.add_predicate_dict({"type": "equals", "key": 'TAXON_KEY', 'value': 44, "matchCase": "false"})
      if k == 'plant':
        gbif_query.add_predicate_dict({"type": "equals", "key": 'TAXON_KEY', 'value': 6, "matchCase": "false"})
      if k == 'invert':
        gbif_query.add_predicate_dict({"type": "equals", "key": 'TAXON_KEY', 'value': 1, "matchCase": "false"})
        gbif_query.add_predicate_dict({"type": "not", "predicate": {"type": "equals", "key": 'TAXON_KEY', 'value': 44}})


      # submit download query
      xx = gbif_query.post_download(userdata.get('GBIF_USER'), userdata.get('GBIF_PWD'))
      # wait for download to be ready
      while True:
        print(f"waiting to get download {xx}...")
        status = occ.download_meta(key = xx)['status']

        if status not in ['PREPARING', 'RUNNING']:  # = not ready yet
            if status == 'SUCCEEDED':
                print(f"Download is ready, getting it")
                output_path = k+"_gbif_observer_potential_pseudoabs"
                if not os.path.exists(output_path): # get rid of any previous downloads for this run
                  os.mkdir(output_path)

                occ.download_get(xx, output_path)
            else:
                print(f"Status is {status}, why?")
                print(occ.download_meta(key = xx))
            break

        sleep(SLEEP_DURATION)

    print("finished with ", k)

/tmp/ipython-input-357306494.py:25: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_master = pd.concat([df_master, df])


downloading  fish
finished with  fish


/tmp/ipython-input-357306494.py:19: DtypeWarning: Columns (14,16,39) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(zf.open(base_name+'.csv'), sep='\t')
/tmp/ipython-input-357306494.py:19: DtypeWarning: Columns (14,16) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(zf.open(base_name+'.csv'), sep='\t')
/tmp/ipython-input-357306494.py:19: DtypeWarning: Columns (14,16,38) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(zf.open(base_name+'.csv'), sep='\t')
/tmp/ipython-input-357306494.py:19: DtypeWarning: Columns (14,16,39) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(zf.open(base_name+'.csv'), sep='\t')
/tmp/ipython-input-357306494.py:19: DtypeWarning: Columns (14,16) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(zf.open(base_name+'.csv'), sep='\t')
/tmp/ipytho

downloading  plant
finished with  plant


/tmp/ipython-input-357306494.py:19: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(zf.open(base_name+'.csv'), sep='\t')


downloading  invert
chunk 3.0/41:  aripla
waiting to get download 0009431-260221153910048...
waiting to get download 0009431-260221153910048...
waiting to get download 0009431-260221153910048...
waiting to get download 0009431-260221153910048...
waiting to get download 0009431-260221153910048...
waiting to get download 0009431-260221153910048...
waiting to get download 0009431-260221153910048...
waiting to get download 0009431-260221153910048...
waiting to get download 0009431-260221153910048...
waiting to get download 0009431-260221153910048...
waiting to get download 0009431-260221153910048...
waiting to get download 0009431-260221153910048...
waiting to get download 0009431-260221153910048...
waiting to get download 0009431-260221153910048...
waiting to get download 0009431-260221153910048...
waiting to get download 0009431-260221153910048...
waiting to get download 0009431-260221153910048...
waiting to get download 0009431-260221153910048...
waiting to get download 0009431-26022115

TimeoutException: Requesting secret GBIF_USER timed out. Secrets can only be fetched when running from the Colab UI.

In [ ]:
# process potential pseudoabsences
import zipfile

pabs = {}
for k in my_vars:
  # read zip file into dataframe
  output_path = k+"_gbif_observer_potential_pseudoabs"
  files = os.listdir(output_path)
  df_master = pd.DataFrame()
  flag = False

  for filename in files:


    file_path = os.path.join(output_path, filename)
    base_name, extension = os.path.splitext(filename)
    zf = zipfile.ZipFile(file_path)
    df = pd.read_csv(zf.open(base_name+'.csv'), sep='\t')

    if not flag:
        df_master = df
        flag = True
    else:
        df_master = pd.concat([df_master, df])

  pabs[k] = df_master

  # drop obs of target species




from pygbif import species as species
from pygbif import occurrences as occ
from pygbif.occurrences.download import GbifDownload

# add your gbif.org username, password and contact email for download notices to Colab's 'Secrets'
# toggle notebook access on for all three
%env GBIF_USER=userdata.get('GBIF_USER')
%env GBIF_PWD = userdata.get('GBIF_PWD')
%env GBIF_EMAIL = userdata.get('GBIF_EMAIL')


# drop obs of target species
def getskey(z):
  return species.name_backbone(z)['usageKey']

for i, (k, v) in enumerate(my_vars.items()):
  splist = v['sci_name'].sum() # make list of all species inc. alternate scientific names
  #splist = list(set(splist))
  spkeys = [ getskey(x) for x in splist ]
  spkeys = list(map(str, spkeys))

  pabs[k] = pabs[k].loc[~pabs[k]['taxonKey'].isin(spkeys)]
  pabs[k] = pabs[k].loc[~pabs[k]['speciesKey'].isin(spkeys)]


  pabs[k].reset_index(drop=True,inplace=True)


KeyError: "There is no item named 'Copy of 0003979-250123221155621.csv' in the archive"

In [ ]:
# intersect with lakes
import geopandas as gpd
# lake polygone
lakes = gpd.read_file("/content/drive/My Drive/iedrr/NHDWaterbody_V2_LakePondReservoir_GLStates.json")
utm = lakes.estimate_utm_crs()
lakes = lakes.to_crs(utm)

for i, (k, v) in enumerate(my_vars.items()):
  print(k)
  if type(pabs[k]) == list:
    continue
  else:
    pabs_gdf = gpd.GeoDataFrame(pabs[k], geometry=gpd.points_from_xy(pabs[k]['decimalLongitude'], pabs[k]['decimalLatitude']), crs="EPSG:4326")
    pabs_gdf = pabs_gdf.to_crs(utm)
    print(len(pabs[k]))

    buffer_distance = 500  # meters

    # Create a buffer around each point

    joined_lakes = gpd.sjoin(lakes, pabs_gdf, predicate='dwithin', distance=buffer_distance)
    joined_lakes.rename(columns={"eventDate":"obs_date"}, inplace=True)
    joined_lakes['obs_date'] = joined_lakes['obs_date'].str.slice(0, 10)
    joined_lakes['obs_date'] = pd.to_datetime(joined_lakes['obs_date'], format='mixed')
    # Drop rows with invalid dates
    joined_lakes = joined_lakes.dropna(subset='obs_date')
    joined_lakes.reset_index(drop=True,inplace=True)

    pabs[k] = joined_lakes[['Permanent_', 'obs_date']]

fish
1412749
plant
689565
invert
2232407


# GLANSIS

In [ ]:
import requests

# get GLANSIS's zipped csv of all data
url = "https://nas.er.usgs.gov/ipt/archive.do?r=nas_glansis"
filename = "GLANSIS_{}.zip".format(today)  # Choose a name for the downloaded file

response = requests.get(url)

if response.status_code == 200:
  with open(filename, "wb") as f:
    f.write(response.content)
  print("Zip file downloaded successfully.")
else:
  print("Failed to download the zip file.")
  # if dl fails, use most recent?
  #from pathlib import Path
  #DIR = Path("drive/My Drive/iedrr")
  #PATTERN = r'GLANSIS_*.zip'
  #latest_file = max(DIR.glob(PATTERN), key=lambda f: f.stat().st_ctime)

# open the zipped file
zf = zipfile.ZipFile(filename)
df = pd.read_csv(zf.open('occurrence.txt'), sep='\t')

df['eventDate'] = pd.to_datetime(df['eventDate'], format="%Y-%m-%d", errors='coerce')
# Drop rows with invalid dates
df = df.dropna(subset='eventDate')

#QAQC
df = df.loc[(df['decimalLatitude'] >= 36.917) & (df['decimalLatitude'] <= 49.612) & (df['decimalLongitude'] >= -100.551) & (df['decimalLongitude'] <= -71.79)] # bounding box
df = df.loc[df['eventDate']>datetime.datetime(1970,1,1)] # drop old data
df = df.loc[df['georeferenceRemarks'] != "Centroid"] # drop obs with poor coordinates
df.reset_index(drop=True,inplace=True)

glansis_obs = {}


for i, (k, v) in enumerate(my_vars.items()):
  splist = v['sci_name'].sum() # make list of all species inc. alternate scientific names
  hightax = []
  for item in splist:
    if len(item.split()) == 1:
      hightax.append(item)

  # drop nontarget species
  ndf = df[(df["scientificName"].isin(splist)) | (df["genus"].isin(hightax))]
  ndf.reset_index(drop=True)

  ndf = ndf[['id', 'scientificName', 'eventDate', 'decimalLatitude', 'decimalLongitude']]
  ndf.rename(columns={'id':'uid', "scientificName":"sci_name", "eventDate":"obs_date", "decimalLatitude":"lat_dec", "decimalLongitude":"lon_dec"}, inplace=True)
  ndf["source"] = "GLANSIS"
  glansis_obs[k] = ndf

Zip file downloaded successfully.


/tmp/ipython-input-820630311.py:23: DtypeWarning: Columns (10,11,12,14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(zf.open('occurrence.txt'), sep='\t')


# MISIN

In [ ]:
!pip install esri2gpd
import esri2gpd


In [ ]:
# MISIN observations layer updated daily
url = "https://services.arcgis.com/uHAHKfH1Z5ye1Oe0/arcgis/rest/services/misin_database_obs/FeatureServer/0"

misin_obs = {}
misin_reporters = {}

# convert esri date to datetime
def convert_esri_date(row):
    """Converts an esriFieldTypeDate value to a Python datetime object."""
    return datetime.datetime.fromtimestamp(row["DAY"] / 1000)  # Divide by 1000 to get seconds

for i, (k, v) in enumerate(my_vars.items()):
  print(k)
  # separate list of genera
  splist = v['sci_name'].sum() # make list of all species inc. alternate scientific names

  # separate out genera with no species epithet
  hightax = []
  for item in splist:
    if len(item.split()) == 1:
      hightax.append(item)
  if "Elodea densa" in splist:
    splist.append("Egeria densa") # MISIN is using Egeria densa instead of Elodea densa
  genus = [x.split()[0] for x in splist]

  gdf = esri2gpd.get(url, fields=['RECORDID', 'OBSERVER', 'DAY', 'LATITUDE', 'LONGITUDE', 'GENUS', 'SPECIES', 'VERIFIED'], where=f"GENUS IN {tuple(genus)}")
  gdf.dropna(subset=['DAY'], inplace=True)
  gdf["DAY"] = gdf.apply(convert_esri_date, axis=1)
  #QAQC
  gdf = gdf[gdf["VERIFIED"] == 2]  # for field VERIFIED, 2 = 'Trusted Source', only keep these
  gdf = gdf.loc[(gdf['LATITUDE'] >= 36.917) & (gdf['LATITUDE'] <= 49.612) & (gdf['LONGITUDE'] >= -100.551) & (gdf['LONGITUDE'] <= -71.79)] # bounding box
  gdf = gdf.loc[gdf['DAY']>datetime.datetime(1970,1,1)] # drop old data
  # drop nontarget species
  gdf["sci_name"] = gdf["GENUS"] + " " + gdf["SPECIES"]
  gdf_targets = gdf[(gdf["sci_name"].isin(splist)) | (gdf["GENUS"].isin(hightax))]
  gdf_targets.reset_index(drop=True,inplace=True)
  # get potential pseudoabsences
  misin_observers = gdf_targets['OBSERVER'].unique()
  print(len(misin_observers), 'observers')
  if len(misin_observers) > 30:
    obs_chunks = []
    start = 0
    end = len(misin_observers)
    step = 30
    for i in range(start, end, step):
      x = i
      observers_chunk = misin_observers[x:x+step]
      if len(observers_chunk) > 1:
        whereclause = f"OBSERVER IN {tuple(observers_chunk)}"
      else:
        whereclause = f"OBSERVER IN ('"+observers_chunk+"')"
      misin_pseudoabs = esri2gpd.get(url, fields=['RECORDID', 'OBSERVER', 'DAY', 'LATITUDE', 'LONGITUDE', 'GENUS', 'SPECIES', 'VERIFIED'], where=whereclause)
      misin_pseudoabs["sci_name"] = misin_pseudoabs["GENUS"] + " " + misin_pseudoabs["SPECIES"]
      misin_pseudoabs.dropna(subset=['DAY'], inplace=True)
      misin_pseudoabs["DAY"] = misin_pseudoabs.apply(convert_esri_date, axis=1)
      misin_pseudoabs = misin_pseudoabs.loc[(misin_pseudoabs['LATITUDE'] >= 36.917) & (misin_pseudoabs['LATITUDE'] <= 49.612) & (misin_pseudoabs['LONGITUDE'] >= -100.551) & (misin_pseudoabs['LONGITUDE'] <= -71.79)] # bounding box
      misin_pseudoabs = misin_pseudoabs.loc[misin_pseudoabs['DAY']>datetime.datetime(1970,1,1)] # drop old data
      misin_pseudoabs = pd.merge(misin_pseudoabs,gdf_targets, indicator=True, how='outer').query('_merge=="left_only"').drop('_merge', axis=1)
      obs_chunks.append(misin_pseudoabs)
    misin_pseudoabs = pd.concat(obs_chunks)
  else:
    misin_pseudoabs = esri2gpd.get(url, fields=['RECORDID', 'OBSERVER', 'DAY', 'LATITUDE', 'LONGITUDE', 'GENUS', 'SPECIES', 'VERIFIED'], where=f"OBSERVER IN {tuple(misin_observers)}")
    misin_pseudoabs.dropna(subset=['DAY'], inplace=True)
    misin_pseudoabs["DAY"] = misin_pseudoabs.apply(convert_esri_date, axis=1)
    misin_pseudoabs = misin_pseudoabs.loc[(misin_pseudoabs['LATITUDE'] >= 36.917) & (misin_pseudoabs['LATITUDE'] <= 49.612) & (misin_pseudoabs['LONGITUDE'] >= -100.551) & (misin_pseudoabs['LONGITUDE'] <= -71.79)] # bounding box
    misin_pseudoabs = misin_pseudoabs.loc[misin_pseudoabs['DAY']>datetime.datetime(1970,1,1)] # drop old data
    misin_pseudoabs = pd.merge(misin_pseudoabs,gdf_targets, indicator=True, how='outer').query('_merge=="left_only"').drop('_merge', axis=1)

  gdf_targets.drop(['geometry', 'OBSERVER', 'GENUS', 'SPECIES', 'VERIFIED'], axis=1, inplace=True)
  gdf_targets.rename(columns={'RECORDID':'uid', "sci_name":"sci_name", "DAY":"obs_date", "LATITUDE":"lat_dec", "LONGITUDE":"lon_dec"}, inplace=True)
  gdf_targets["source"] = "MISIN"
  gdf_targets['uid'] = gdf_targets['uid'].astype(str)

  misin_pseudoabs.drop(['geometry', 'OBSERVER', 'GENUS', 'SPECIES', 'VERIFIED'], axis=1, inplace=True)
  misin_pseudoabs.rename(columns={'RECORDID':'uid', "sci_name":"sci_name", "DAY":"obs_date", "LATITUDE":"lat_dec", "LONGITUDE":"lon_dec"}, inplace=True)
  misin_pseudoabs['uid'] = misin_pseudoabs['uid'].astype(str)

  misin_obs[k] = gdf_targets
  misin_reporters[k] = misin_pseudoabs

fish
35 observers


/usr/local/lib/python3.12/dist-packages/esri2gpd/core.py:91: UserWarning: Long download time — total download will require 235 separate requests
  warnings.warn(
/tmp/ipython-input-509132713.py:69: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gdf_targets.drop(['geometry', 'OBSERVER', 'GENUS', 'SPECIES', 'VERIFIED'], axis=1, inplace=True)
/tmp/ipython-input-509132713.py:70: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gdf_targets.rename(columns={'RECORDID':'uid', "sci_name":"sci_name", "DAY":"obs_date", "LATITUDE":"lat_dec", "LONGITUDE":"lon_dec"}, inplace=True)
/usr/local/lib/python3.12/dist-packages/geopandas/geodat

plant


/usr/local/lib/python3.12/dist-packages/esri2gpd/core.py:91: UserWarning: Long download time — total download will require 11 separate requests
  warnings.warn(


155 observers


/usr/local/lib/python3.12/dist-packages/esri2gpd/core.py:91: UserWarning: Long download time — total download will require 241 separate requests
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/esri2gpd/core.py:91: UserWarning: Long download time — total download will require 11 separate requests
  warnings.warn(
/tmp/ipython-input-509132713.py:69: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gdf_targets.drop(['geometry', 'OBSERVER', 'GENUS', 'SPECIES', 'VERIFIED'], axis=1, inplace=True)
/tmp/ipython-input-509132713.py:70: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gdf_targets.rename(columns={'RECORDID':'ui

invert
67 observers


/usr/local/lib/python3.12/dist-packages/esri2gpd/core.py:91: UserWarning: Long download time — total download will require 228 separate requests
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/esri2gpd/core.py:91: UserWarning: Long download time — total download will require 11 separate requests
  warnings.warn(
/tmp/ipython-input-509132713.py:69: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gdf_targets.drop(['geometry', 'OBSERVER', 'GENUS', 'SPECIES', 'VERIFIED'], axis=1, inplace=True)
/tmp/ipython-input-509132713.py:70: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gdf_targets.rename(columns={'RECORDID':'ui

In [ ]:
# intersect pseudoobs with lakes
import geopandas as gpd
# lake polygone
if not 'lakes' in locals():
  lakes = gpd.read_file("/content/drive/My Drive/iedrr/NHDWaterbody_V2_LakePondReservoir_GLStates.json")
  print(lakes.crs)
  utm = lakes.estimate_utm_crs()
  lakes = lakes.to_crs(utm)
  print(lakes.crs)

misin_abs = {}

for i, (k, v) in enumerate(my_vars.items()):
  print(k)
  if type(misin_reporters[k]) == list:
    continue
  else:
    misin_gdf = gpd.GeoDataFrame(misin_reporters[k], geometry=gpd.points_from_xy(misin_reporters[k]['lon_dec'], misin_reporters[k]['lat_dec']), crs="EPSG:4326")
    misin_gdf = misin_gdf.to_crs(utm)
    print(misin_gdf.crs)
    print(len(misin_reporters[k]))

    buffer_distance = 500  # meters

    # Create a buffer around each point

    joined_lakes = gpd.sjoin(lakes, misin_gdf, predicate='dwithin', distance=buffer_distance)

    joined_lakes = joined_lakes.dropna(subset='obs_date')
    joined_lakes.reset_index(drop=True,inplace=True)

    misin_abs[k] = joined_lakes[['Permanent_', 'obs_date']]

EPSG:4326
EPSG:32616
fish
EPSG:32616
462800
plant
EPSG:32616
555221
invert
EPSG:32616
473724


# iMapInvasives

In [ ]:
!pip install esri2gpd
import esri2gpd
# in GL states, iMapInvasives obs are almost exclusively in PA and NY


80 observers
['Robert Angyal - 2624' 'David Pawlowski - 3199' 'Stephen Walsh - 3173'
 'Denise Clay - 2348' 'Amy Mahar - 3380' 'Stacy Furgal - 3099'
 'Melissa Cohen - 2628' 'Brittney Rogers - 4361' 'Jeremy Dietrich - 3772'
 'Doug Carlson - 2623' 'Scott Wells - 2656' 'Robert Schmidt - 3343'
 'Emily Zollweg - 2659' 'Marcus Rosten - 4808' 'Brad Mudrzynski - 4546'
 'SWFDB DEC Fisheries - 5612' 'John Farrell - 2660' 'Lucy Nuessle - 5893'
 'Public Reports - 6034' 'USGS-NAS Database USGS-NAS Database - 6414'
 'Kirkland Nagy - 7914' 'Alex Parry - 9741' 'John Cooper - 7454'
 'Eric Chapman - 12691' 'Jennifer Dzimiela - 12864'
 'Robert Morgan - 12423'
 'Representative(s) from Gannon University - 12902'
 'Representative(s) from PA Fish & Boat Commission - 12905'
 'Mark Lethaby - 12659' 'Jim Grazio - 12566']
['Nate Irwin - 12682' 'Boris Kitevski - 13167' 'Rebecah Ford - 13589'
 'Michael Hosack - 12669' 'Robert Wnuk - 12923' 'Jay Stauffer - 13678'
 'USGS-NAS Database - 13574' 'Nick Macelko - 13476'
 

/usr/local/lib/python3.11/dist-packages/esri2gpd/core.py:91: UserWarning: Long download time — total download will require 61 separate requests
  warnings.warn(


892 observers
['Robert L. Johnson - 2251' 'Stacy Furgal - 3099'
 'Nancy Davis-Ricci - 2196' 'Janet Andersen - 2359'
 'Meghan Johnstone - 3208' 'Mike McHale - 3108' 'Mike Goehle - 2611'
 'Hilary Smith - 2303' 'Denise Clay - 2348' 'Greg Chapman - 3100'
 'Emily Sheridan - 3107' 'Paul Marangelo - 15165' 'Jeff Sann - 3106'
 'Scott Kishbaugh - 2262' "Heidi O'Riordan - 3281"
 'Samantha Knowlden - 2433' 'Meg Wilkinson - 2084' 'Nancy Engel - 2511'
 'Chuck Nichols - 3330' 'William Hoffman - 3168' 'Julie Nace - 3498'
 'Kelly McDonald - 4071' 'Gail Meyer - 3182' 'Lawrence Eichler - 2255'
 'Willard Harman - 3133' 'Heather Ruel - 2587' 'Alexis Alvey - 3283'
 'Chris Lajewski - 2302' 'Alyssa Reid - 3025' 'Michael Goethle - 2121']
['Lindsay (Jody) Hoyt - 3750' 'Kevin Jennings - 3033'
 'Marisa Kwoczka - 2529' 'Holly Menninger - 2247' 'Ariana Newell - 2377'
 'Donna Vogler - 2395' 'Walt Nelson - 2283' 'David Newman - 4221'
 'John Davis - 2452' 'Erik Posner - 3187' 'Kersten Laveroni - 3878'
 'Meredith Tayl

<ipython-input-59-9ad4f4ce9418>:67: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rdf.rename(columns={'present_species_id':'uid', "scientific_name":"sci_name", "observation_date":"obs_date"}, inplace=True)
/usr/local/lib/python3.11/dist-packages/geopandas/geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
/usr/local/lib/python3.11/dist-packages/geopandas/geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value i

204 observers
['Meghan Brown - 3011' 'Dave Strayer - 3143' 'David Thompson - 13893'
 'Alexander Karatayev - 3510' 'Mike McHale - 3108' 'Donna Vogler - 2395'
 'Emily DeBolt - 17819' 'Amy Mahar - 3380' 'Mike Ellrott - 3236'
 'Doc Bayne - 3861' 'Bruce Natale - 2751' 'Nancy Engel - 2511'
 'Kristen Rohne - 2740' 'Jeff Corser - 2195' 'Meg Modley - 3258'
 'Luke Myers - 3977' 'Bradley Bowers - 3127' 'Julie A. Lundgren - 2063'
 'Derek Conant - 3670' 'unknown Austerman - 3518' 'Jesse Jaycox - 2552'
 'Willard Harman - 3133' 'Russell Nemecek - 4088' 'Doug Carlson - 2623'
 'Casey Holzworth - 2559' 'Cody Mendoza - 3396' 'Jordan Youngmann - 4446'
 'Edward Levri - 12746' 'Robert Schmidt - 3343' 'Jeremy Dietrich - 3772']
['Bob Daniels - 3252' 'Diana Heitzman - 3512' 'Cameron Jenness - 4661'
 'Francine Stayter - 4306' 'Damian Griffin - 3488'
 'Michele Wunderlich - 2731' 'Elizabeth MacEwen - 5155'
 'Fredric Dunlap - 4196' 'Samantha Olsen - 5309' 'Eric Holmlund - 2304'
 'John Slyer - 5430' 'Tom Brooking -

<ipython-input-59-9ad4f4ce9418>:67: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rdf.rename(columns={'present_species_id':'uid', "scientific_name":"sci_name", "observation_date":"obs_date"}, inplace=True)
/usr/local/lib/python3.11/dist-packages/geopandas/geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
/usr/local/lib/python3.11/dist-packages/geopandas/geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value i

In [ ]:
url = "https://imapinvasives.natureserve.org/arcgis/rest/services/public_presence/MapServer/4" # presences
url2 = "https://imapinvasives.natureserve.org/arcgis/rest/services/public_approximate_presence/MapServer/4" # approx coordinates presences

imap_obs = {}
imap_recorders = {}

import geopandas as gpd

# convert esri date to datetime
def convert_esri_date(row):
    """Converts an esriFieldTypeDate value to a Python datetime object."""
    return datetime.datetime.fromtimestamp(row["observation_date"] / 1000)  # Divide by 1000 to get seconds

for i, (k, v) in enumerate(my_vars.items()):
  # separate list of genera
  splist = v['sci_name'].sum() # make list of all species inc. alternate scientific names
  # separate out genera with no species epithet
  hightax = []
  for item in splist:
    if len(item.split()) == 1:
      hightax.append(item)

  genus = [x.split()[0] for x in splist]

  gdf = esri2gpd.get(url, fields=['present_species_id', 'observer_name', 'observation_date', 'jurisdiction', 'genus', 'scientific_name'], where=f"genus IN {tuple(genus)} AND jurisdiction IN ('New York', 'Pennsylvania', 'Michigan', 'Ohio')")
  gdf2 = esri2gpd.get(url2, fields=['present_species_id', 'observer_name', 'observation_date', 'jurisdiction', 'genus', 'scientific_name'], where=f"genus IN {tuple(genus)} AND jurisdiction IN ('New York', 'Pennsylvania', 'Michigan', 'Ohio')")
  rdf = gpd.GeoDataFrame(pd.concat([gdf, gdf2], ignore_index=True), crs=gdf.crs)

  rdf["observation_date"] = rdf.apply(convert_esri_date, axis=1)
  rdf["observation_date"] = pd.to_datetime(rdf["observation_date"], errors='coerce')
  #QAQC
  rdf = rdf.loc[rdf['observation_date']>datetime.datetime(1970,1,1)] # drop old data

  # get potential pseudoabsences
  imi_observers = rdf['observer_name'].unique()
  print(len(imi_observers), 'observers')
  if len(imi_observers) > 30:
    obs_chunks = []
    start = 0
    end = len(imi_observers)
    step = 30
    for i in range(start, end, step):
      x = i
      observers_chunk = imi_observers[x:x+step]
      if len(observers_chunk) > 1:
        whereclause = f"observer_name IN {tuple(observers_chunk)} AND jurisdiction IN ('New York', 'Pennsylvania', 'Michigan', 'Ohio')"
      else:
        whereclause = f"OBSERVER IN ('"+observers_chunk+"') AND jurisdiction IN ('New York', 'Pennsylvania', 'Michigan', 'Ohio')"
  imi_pseudoabs = esri2gpd.get(url, fields=['present_species_id', 'observer_name', 'observation_date', 'jurisdiction', 'genus', 'scientific_name'], where=whereclause)
  imi_pseudoabs["observation_date"] = imi_pseudoabs.apply(convert_esri_date, axis=1)
  imi_pseudoabs["observation_date"] = pd.to_datetime(imi_pseudoabs["observation_date"], errors='coerce')
  imi_pseudoabs = imi_pseudoabs.loc[imi_pseudoabs['observation_date']>datetime.datetime(1970,1,1)] # drop old data
  imi_pseudoabs = pd.merge(imi_pseudoabs,rdf, indicator=True, how='outer').query('_merge=="left_only"').drop('_merge', axis=1)
  imi_pseudoabs.rename(columns={'present_species_id':'uid', "scientific_name":"sci_name", "observation_date":"obs_date"}, inplace=True)
  imi_pseudoabs['uid'] = imi_pseudoabs['uid'].astype(str)
  imi_pseudoabs['lat_dec'] = imi_pseudoabs['geometry'].y.astype(float)
  imi_pseudoabs['lon_dec'] = imi_pseudoabs['geometry'].x.astype(float)
  imi_pseudoabs.drop(['geometry', 'observer_name', 'jurisdiction', 'genus'], axis=1, inplace=True)
  # drop nontarget species
  rdf = rdf[(rdf["scientific_name"].isin(splist)) | (rdf["genus"].isin(hightax))]
  rdf.reset_index(drop=True,inplace=True)
  rdf.rename(columns={'present_species_id':'uid', "scientific_name":"sci_name", "observation_date":"obs_date"}, inplace=True)
  rdf['uid'] = rdf['uid'].astype(str)
  rdf['lat_dec'] = rdf['geometry'].y.astype(float)
  rdf['lon_dec'] = rdf['geometry'].x.astype(float)
  rdf.drop(['geometry', 'observer_name', 'jurisdiction', 'genus'], axis=1, inplace=True)
  rdf["source"] = "iMapInvasives"
  # all obs in layer have been confirmed

  imap_obs[k] = rdf
  imap_recorders[k] = imi_pseudoabs


87 observers


/usr/local/lib/python3.12/dist-packages/esri2gpd/core.py:91: UserWarning: Long download time — total download will require 64 separate requests
  warnings.warn(


987 observers
220 observers


In [ ]:
# intersect pseudoobs with lakes
import geopandas as gpd
# lake polygone
if lakes is None:
  lakes = gpd.read_file("/content/drive/My Drive/iedrr/NHDWaterbody_V2_LakePondReservoir_GLStates.json")
  print(lakes.crs)
  utm = lakes.estimate_utm_crs()
  lakes = lakes.to_crs(utm)
  print(lakes.crs)

imap_abs = {}

for i, (k, v) in enumerate(my_vars.items()):
  print(k)
  if type(imap_recorders[k]) == list:
    continue
  else:
    gdf = gpd.GeoDataFrame(imap_recorders[k], geometry=gpd.points_from_xy(imap_recorders[k]['lon_dec'], imap_recorders[k]['lat_dec']), crs="EPSG:4326")
    gdf = gdf.to_crs(utm)
    print(gdf.crs)
    print(len(imap_recorders[k]))

    buffer_distance = 500  # meters

    # Create a buffer around each point

    joined_lakes = gpd.sjoin(lakes, gdf, predicate='dwithin', distance=buffer_distance)

    joined_lakes = joined_lakes.dropna(subset='obs_date')
    joined_lakes.reset_index(drop=True,inplace=True)

    imap_abs[k] = joined_lakes[['Permanent_', 'obs_date']]

fish
EPSG:32616
1964
plant
EPSG:32616
453
invert
EPSG:32616
4192


# EDDMapS

In [ ]:
from google.colab import userdata
import requests
import json
from pandas import json_normalize

edd_obs = {}

# log in to EDDMapS API
s = requests.session()

login_url = 'https://api.bugwoodcloud.org/v2/login'
headers = {
    "accept": "application/json",
    "Content-Type": "application/json"
}
resp = s.post(login_url, headers=headers, json={"email": userdata.get('EDDMAPS_USER'), "password": userdata.get('EDDMAPS_PWD')})

# resp.raise_for_status()

# can't search occurrence data using scientific names directly, have to get IDs first
subj_url = "https://api.bugwoodcloud.org/v2/subject"
sparams = dict()
sparams["searchon"] = "ScientificName"

for i, (k, v) in enumerate(my_vars.items()):
  print("starting ",k)

  namelist = []

  splist = v['sci_name'].sum() # make list of all species inc. alternate scientific names

  for name in splist:
    #print(name)
    # have to drop space between genus and species
    name = name.replace(" ", "")
    sparams["search"] = name
    r = s.get(subj_url, params=sparams)

    if r.status_code == 200:
      # Parse the JSON response
      data = json.loads(r.text)
      df = json_normalize(data)
      if df.empty:
        print("No results for",name)
        continue
      else:
        # specify fields to extract
        subfields = ["subjectid", "namepart1", "namepart2"]
        df = df[subfields]
        # clean up extra taxa that contain snakehead genus name
        if k == "fish":
          mask = df[df.apply(lambda row: row.str.contains('channa', case=False).any(), axis=1)]
          if not mask.empty:
            mask = mask.loc[mask['namepart1'] != "Channa"]
            df = pd.merge(df,mask, indicator=True, how='outer') \
              .query('_merge=="left_only"') \
              .drop('_merge', axis=1)
        #print (df)
        namelist.append(df)

    else:
        print("Error: ", r.status_code)

  namelist = pd.concat(namelist, ignore_index=True)

  subjs = ','.join(namelist['subjectid'].astype(str)) #comma-separated list of subject ids for species of interest

  # get occurrence data
  occ_url = 'https://api.bugwoodcloud.org/v2/occurrence'
  params = dict()
  params["subjectid"] = subjs
  # "scientificname" is not a valid param, need to use subjectids
  params["sortorder"] = "desc"
  params["enddate"] = "01/17/2025"
  params["startdate"] = "01/01/1970"
  params["paging"] = "false"

  statedfs = []

  for state in ["17","18","26","27","36","39","42","55"]: # FIPS codes for IL, IN, MI, MN, NY, OH, PA, WI
    params["state"] = state
    r2 = s.get(occ_url, params=params) # is there a limit to number of records?

    # Check if the request was successful
    if r2.status_code == 200:
      # Parse the JSON response
      data = json.loads(r2.text)
      df = json_normalize(data)
      # specify fields to extract
      subfields = ["objectid", "scientificname", "observationdate", "latitude_decimal", "longitude_decimal", "recordbasis", "identificationcredibility"]
      if not df.empty:
        df = df[subfields]
        statedfs.append(df)
      else:
        print("No ",k," results for ",state)

    else:
        print("Error: ", r.status_code)

  cdf = pd.concat(statedfs, ignore_index=True)
  # QAQC
  cdf = cdf[cdf.recordbasis != "Preserved Specimen"]
  keeps = ["Credible", "Verified"]
  cdf = cdf[cdf['identificationcredibility'].isin(keeps)]
  cdf.dropna(inplace=True)
  cdf.reset_index(drop=True,inplace=True)
  cdf.drop(['recordbasis', 'identificationcredibility'], axis=1, inplace=True)
  cdf.rename(columns={'objectid':'uid', "scientificname":"sci_name", "observationdate":"obs_date", "latitude_decimal":"lat_dec", "longitude_decimal":"lon_dec"}, inplace=True)
  cdf["source"] = "EDDMapS"
  cdf['uid'] = cdf['uid'].astype(str)
  cdf['obs_date'] = pd.to_datetime(cdf['obs_date'])
  print (cdf.shape[0],k," records")

  edd_obs[k] = cdf



starting  fish
No  fish  results for  39
No  fish  results for  42
28 fish  records
starting  plant
No results for Pontederiaazurea
40 plant  records
starting  invert
22 invert  records


# Combine

In [ ]:

# is there a lake within a km of the obs

nhd3d = HP3D("waterbody")
nhdp_wb = WaterData("nhdwaterbody")
nhdp_mr = WaterData("nhdflowline_network")
# pynhd.pynhd.NHD has waterbody_hr but only WaterData has bydistance function
eck4 = "+proj=eck4 +lon_0=0 +x_0=0 +y_0=0 +datum=WGS84 +units=m +no_defs"
rad = 5e2 # find reaches within 500 m


In [ ]:
count_lists = []
for k in my_vars:
  print(k)
  # combine obs from all sources that you ran the code for
  dflist = []
  try:
    dflist.append(gbif_obs[k])
  except:
    pass
  try:
    dflist.append(glansis_obs[k])
  except:
    pass
  try:
    dflist.append(imap_obs[k])
  except:
    pass
  try:
    dflist.append(edd_obs[k])
  except:
    pass
  try:
    dflist.append(misin_obs[k])
  except:
    pass
  for x in dflist:
    if pd.api.types.is_dtype_equal(x['obs_date'].dtype, "object"):
      x['obs_date'] = pd.to_datetime(x['obs_date'])
    if pd.api.types.is_dtype_equal(x['obs_date'].dtype, "datetime64[ns, UTC]"):
      x['obs_date'] = x['obs_date'].dt.tz_localize(None)
    x['obs_date'] = x['obs_date'].dt.normalize()
    #print(x['obs_date'].dtype)
  df = pd.concat(dflist, ignore_index=True)
  df.reset_index(drop=True, inplace=True)

  # use consistent scientific name for Brazilian waterweed, water hyacinths todo: generalize this
  df.replace({'sci_name':{'Elodea densa':'Egeria densa', 'Eichhornia crassipes':'Pontederia crassipes', 'Eichhornia azurea':'Pontederia azurea'}}, inplace = True)

  # final QAQC before counts
  #drop observations whose coordinates are the location of an institution
  rcnt = df.shape[0]
  print(rcnt, " recs before QAQC")
  df["mindist"] = df.apply(calculate_min, axis=1)
  df = df[df["mindist"] > 0.00001] # buffer in degrees
  df.reset_index(drop=True, inplace=True)
  rcnt2 = df.shape[0]
  if rcnt2 < rcnt:
    print(rcnt - rcnt2, " recs at institution lat/longs were dropped")
  # drop obs if lat and long are the same
  df = df[df['lat_dec'] != df['lon_dec']]
  rcnt3 = df.shape[0]
  if rcnt3 < rcnt2:
    print(rcnt2 - rcnt3, " recs where lat = lon were dropped")
  df.dropna(inplace=True)
  rcnt4 = df.shape[0]
  if rcnt4 < rcnt3:
    print(rcnt3 - rcnt4, " recs with nans were dropped")
  print(rcnt4, " recs after QAQC")
  if extent == "states":
  # clip to the Great Lakes states
    import geopandas as gpd
    cwd = os.getcwd()

    # Get US state polygons
    states = pygris.states()
    # Great Lakes states
    states = states[states['NAME'].isin(['Illinois', 'Indiana', 'Michigan', 'Minnesota', 'New York', 'Ohio', 'Pennsylvania', 'Wisconsin'])]

    gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df['lon_dec'], df['lat_dec']))

    # assume WGS 84 for these point obs, convert to match crs for states
    gdf.set_crs(epsg=4326, inplace=True)
    gdf.to_crs(epsg=4269, inplace=True)

    #Output
    dfGL = gpd.sjoin(gdf, states, predicate='within')
    df = dfGL.drop(['geometry',	'index_right',	'STATEFP',	'STATENS',	'GEOIDFQ',	'GEOID',	'STUSPS',	'NAME',	'LSAD',	'ALAND',	'AWATER'], axis=1)
    df.reset_index(drop=True,inplace=True)

  # total recs from each source
  for category, count in df['source'].value_counts().items():
      print(f"{count} total recs from {category}")
      row = [k,category, 'total', count]
      count_lists.append(row)
  # drop duplicates, with a margin of error to account for coordinate rounding

  distance = 0.0001
  xy = df[['lon_dec','lat_dec']].values.tolist()
  cluster = AgglomerativeClustering(n_clusters=None, linkage='single', metric='euclidean', distance_threshold=distance)
  cluster.fit(xy)
  df['group'] = cluster.labels_

  # combine group id, species, date into one microgroup column
  cols = ['group', 'sci_name', 'obs_date']
  df['microgroup'] = df[cols].apply(lambda row: '_'.join(row.values.astype(str)), axis=1)
  # Create a new column with the count of each microgroup
  # if count(microgroup)>1, mark as duplicate
  df['mg_count'] = df.groupby('microgroup')['microgroup'].transform('count')
  df['is_duplicate'] = df['mg_count'] > 1
  # how many recs does each source have that aren't in the other sources
  dfu = df[df['is_duplicate'] != True]
  for category, count in dfu['source'].value_counts().items():
      print(f"{count} unique recs from {category}")
      row = [k,category, 'unique', count]
      count_lists.append(row)
  print(df['is_duplicate'].value_counts())
  print('unique recs: ',df['microgroup'].nunique())
  # Within each microgroup keep the first version of that observation
  df = df.groupby('microgroup', as_index=False).first()
  df.reset_index(drop=True,inplace=True)
  df.drop(['group', 'microgroup', 'mg_count', 'mindist', 'is_duplicate'], axis=1, inplace=True)

  df["taxon"] = k
  # export final obs table for taxon to Drive folder
  cwd = os.getcwd()
  outfile = os.path.join(cwd, k + '_obs_allsources_' + extent + datetime.date.today().strftime('%Y%m%d') + '.csv')
  df.to_csv(outfile, index=False)
  print("exported to ",outfile)


# save total and unique observation counts for each source
count_headers = ['taxon','source', 'type', 'count']
counts_df = pd.DataFrame(count_lists,columns=count_headers)
#counts_df = counts_df.to_frame().reset_index()
allcounts = os.path.join(cwd, 'obs_counts_by_taxon_and_source_' + extent + datetime.date.today().strftime('%Y%m%d') + '.csv')
counts_df.to_csv(allcounts, index=False)

counts_df.drop(['taxon'], axis=1, inplace=True)

counts_df = counts_df.groupby(['source', 'type'])['count'].sum()
counts_df = counts_df.to_frame().reset_index()

counts_df = counts_df.sort_values(by=['source', 'type'])
countfile = os.path.join(cwd, 'obs_counts_by_source_' + datetime.date.today().strftime('%Y%m%d') + '.csv')
counts_df.to_csv(countfile, index=False)
# drop GBIF exports to save space
!rm -rf ./invert_gbif_obs/
!rm -rf ./plant_gbif_obs/
!rm -rf ./fish_gbif_obs/
print("done!")

fish
44665  recs before QAQC
44665  recs after QAQC
Using the default year of 2024
4253 total recs from GBIF
3415 total recs from GLANSIS
2277 total recs from MISIN
1980 total recs from iMapInvasives
18 total recs from EDDMapS
2531 unique recs from GBIF
881 unique recs from GLANSIS
544 unique recs from iMapInvasives
461 unique recs from MISIN
is_duplicate
True     7526
False    4417
Name: count, dtype: int64
unique recs:  7539
exported to  /content/drive/MyDrive/iedrr/speciesobs_20260208/fish_obs_allsources_states20260210.csv
plant
92193  recs before QAQC
92193  recs after QAQC
Using the default year of 2024
34059 total recs from iMapInvasives
19558 total recs from GLANSIS
14272 total recs from MISIN
9965 total recs from GBIF
35 total recs from EDDMapS
13495 unique recs from iMapInvasives
6107 unique recs from MISIN
3358 unique recs from GBIF
477 unique recs from GLANSIS
21 unique recs from EDDMapS
is_duplicate
True     54431
False    23458
Name: count, dtype: int64
unique recs:  46021

In [ ]:
# combine potential pseudoabs lakes
cwd = os.getcwd()

for k in my_vars:
  print(k)
  # combine obs from all sources
  plist = [pabs[k], misin_abs[k], imap_abs[k]]

  for x in plist:
    if pd.api.types.is_dtype_equal(x['obs_date'].dtype, "datetime64[ns, UTC]"):
      x['obs_date'] = x['obs_date'].dt.tz_localize(None)
    x['obs_date'] = x['obs_date'].dt.normalize()
    #print(x['obs_date'].dtype)
  df = pd.concat(plist, ignore_index=True)
  df.reset_index(drop=True, inplace=True)
  df.dropna(inplace=True)

  outfile = os.path.join(cwd, k+'_potential_pseudoabs_lakes' + datetime.date.today().strftime('%Y%m%d') + '.csv')
  df.to_csv(outfile, index=False)


fish
plant
invert


<ipython-input-63-2117b76c5841>:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  x['obs_date'] = x['obs_date'].dt.normalize()
